In [20]:
import os
%pwd


'c:\\Users\\HP\\Desktop\\coding\\kidney-disease-classification'

In [2]:
os.chdir('../')


In [21]:
%pwd


'c:\\Users\\HP\\Desktop\\coding\\kidney-disease-classification'

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data_path: Path
    params_epoch: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [23]:

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [24]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        # Correct dataset directory
        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir
        )

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(
                prepare_base_model.updated_base_model_path
            ),
            training_data_path=Path(training_data),
            params_epoch=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [25]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [26]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=1e-4
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

    def train_valid_generator(self):
        # Note: EfficientNetV2 includes internal rescaling & normalization [0, 255]
        datagenerator_kwargs = dict(
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=tuple(self.config.params_image_size[:2]),
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical"
        )

        valid_datagenerator = (
            tf.keras.preprocessing.image.ImageDataGenerator(
                **datagenerator_kwargs
            )
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=str(self.config.training_data_path),
            subset="validation",
            shuffle=False,
            seed=42,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = (
                tf.keras.preprocessing.image.ImageDataGenerator(
                    rotation_range=20,
                    horizontal_flip=True,
                    width_shift_range=0.1,
                    height_shift_range=0.1,
                    zoom_range=0.1,
                    **datagenerator_kwargs
                )
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=str(self.config.training_data_path),
            subset="training",
            shuffle=True,
            seed=42,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(str(path))

    def train(self):
        # Compute balanced class weights to prevent majority-class collapse
        labels = self.train_generator.classes
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(labels),
            y=labels
        )
        class_weight_dict = dict(enumerate(class_weights))
        print("Calculated class weights:", class_weight_dict)

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.2,
                patience=2,
                min_lr=1e-6,
                verbose=1
            )
        ]

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epoch,
            validation_data=self.valid_generator,
            class_weight=class_weight_dict,
            callbacks=callbacks
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )


In [27]:

try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-10-18 00:14:38,514: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-18 00:14:38,523: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-18 00:14:38,527: INFO: common: created directory at: artifacts]
[2026-10-18 00:14:38,532: INFO: common: created directory at: artifacts\training]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.
Calculated class weights: {0: np.float64(0.8388645552560647), 1: np.float64(0.6129369768586903), 2: np.float64(2.259301270417423), 3: np.float64(1.3627531472359058)}
Epoch 1/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 416s 647ms/step - accuracy: 0.6855 - loss: 0.8505 - val_accuracy: 0.5416 - val_loss: 1.1770 - learning_rate: 1.0000e-04
Epoch 2/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 400s 643ms/step - accuracy: 0.8545 - loss: 0.3903 - val_accuracy: 0.5501 - val_loss: 1.2642 - learning_rate: 1.0000e-04
Epoch 3/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 400s 642ms/step - accuracy: 0.9021 - loss: 0.2633 - val_accu